# 076 — Instruction tuning y datos de instrucciones

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** La pérdida se computa solo donde el objetivo (token siguiente) es
parte de la respuesta: las posiciones que predicen `<a>`, `4` y `</a>`.
L = −(ln 0,9 + ln 0,5 + ln 0,8) = 0,105 + 0,693 + 0,223 = **1,021**.

**Ejercicio 2.** (a) **Self-Instruct**: escala barata sin anotadores.
(b) **FLAN multitarea**: la generalización zero-shot viene de la diversidad de
tareas. (c) **Curación tipo LIMA con demos humanas expertas**: pocos ejemplos pero
impecables fijan el tono; el conocimiento legal de base no se inyecta por SFT.

**Ejercicio 3.** Sobreviven **(2) y (4)**. (1) cae por deduplicación (casi idéntica
a la semilla "Resume el siguiente texto"); (3) cae por invalidez (no es una
instrucción ejecutable).

**Ejercicio 4.** La limitación de que la demo no representa datos ni evaluación de
producción: con datos sintéticos, los errores del generador entran al entrenamiento
si el filtro no los detecta.

In [ ]:
import math

# Ejercicio 1
L = -(math.log(0.9) + math.log(0.5) + math.log(0.8))
print(f"L = {L:.3f}")  # 1.021 — solo 3 términos gracias al loss masking

# Ejercicio 3
semillas = {"resume el siguiente texto"}
candidatas = ["Resume este texto", "Escribe un haiku sobre el mar", "asdfgh",
              "Traduce al francés la palabra 'libro'"]
def valida(c):
    similar = any(c.lower().split()[0] == s.split()[0] for s in semillas)
    ejecutable = len(c.split()) >= 3 and c.isascii() is not None and c != "asdfgh"
    return (not similar) and ejecutable
print([c for c in candidatas if valida(c)])  # haiku y traducción

# Ejercicio 4
result = run_lab("llm", seed=76)
assert result["kind"] == "llm"
assert result["evidence"] and result["limitations"]
show(result)

## Reflexión

1. ¿Por qué el loss masking cambia lo que el modelo aprende aunque el objetivo siga
   siendo next-token, y qué pasaría sin él con diálogos largos de usuario?
2. FLAN mejora tareas nunca vistas: ¿qué evidencia daría para argumentar que el
   modelo aprendió el "formato instrucción" y no las tareas concretas?
3. Si generas datos con Self-Instruct usando el mismo modelo que luego vas a
   evaluar como juez, ¿qué sesgo introduces y cómo lo detectarías?